# Customer Shopping Behavior Analysis
This notebook covers the exploratory data analysis, data cleaning, feature engineering, and database integration of customer shopping behavior data. We will prepare the raw dataset for analysis and export it into a local SQL database for querying.

### Import Libraries and Load Dataset
We start by importing the necessary libraries (like pandas) and loading the customer behavior dataset.

In [16]:
import pandas as pd

# Load the raw customer shopping behavior dataset into a pandas DataFrame
df = pd.read_csv('customer_shopping_behavior.csv')

### Exploratory Data Analysis (EDA)
Let's inspect the data structure, data types, summary statistics, and check for missing values to understand what needs cleaning.

In [18]:
# Display the first 5 rows of the DataFrame to inspect the initial values and structure
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [19]:
# Print a concise summary of the DataFrame, including columns, non-null counts, and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [20]:
# Generate summary statistics for all columns, including categorical/object fields
df.describe(include='all')

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
count,3900.000000,3900.000000,3900,3900,3900,3900.000000,3900,3900,3900,3900,3863.000000,3900,3900,3900,3900,3900.000000,3900,3900
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Blouse,Clothing,NaN,Montana,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2652,171,1737,NaN,96,1755,177,999,NaN,2847,675,2223,2223,NaN,677,584
mean,1950.500000,44.068462,NaN,NaN,NaN,59.764359,NaN,NaN,NaN,NaN,3.750065,NaN,NaN,NaN,NaN,25.351538,NaN,NaN
std,1125.977353,15.207589,NaN,NaN,NaN,23.685392,NaN,NaN,NaN,NaN,0.716983,NaN,NaN,NaN,NaN,14.447125,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,975.750000,31.000000,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1950.500000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.800000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2925.250000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


In [21]:
# Count the number of missing (null) values in each column
df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

### Data Cleaning and Imputation
We identified some missing values in Review Rating. We will impute these values using the median review rating of the corresponding product Category to maintain data consistency.

In [23]:
# Fill missing 'Review Rating' values with the median rating of their respective product 'Category'
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [24]:
# Re-check null counts to verify that all missing values in 'Review Rating' have been resolved
df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

### Feature Engineering and Column Transformation
In this section, we will standardise column names to snake_case, perform binning/discretization on the age variable, and convert categorical purchase frequencies into numerical days for downstream quantitative analysis.

In [26]:
# Standardize column names to snake_case: convert to lowercase and replace spaces with underscores
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ','_')

# Simplify the purchase amount column name for easier reference
df = df.rename(columns={'purchase_amount_(usd)':'purchase_amount'})

In [27]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [28]:
# Define target age category labels
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']

# Split 'age' into 4 equal-sized quantile groups (bins) and assign labels
df['age_group'] = pd.qcut(df['age'], q=4, labels = labels)

In [29]:
df[['age', 'age_group']].head(10)

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


In [30]:
# Create a mapping dictionary to convert categorical frequencies into numerical days
frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

# Create a new numeric column by mapping the 'frequency_of_purchases' column
df['purchase_frequeny_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [31]:
# Display the first 10 rows to confirm the mapping from categorical frequency to days was successful
df[['purchase_frequeny_days', 'frequency_of_purchases']].head(10)

,purchase_frequeny_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [32]:
# Inspect the first 10 rows of 'discount_applied' and 'promo_code_used' side-by-side
df[['discount_applied', 'promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [33]:
# Check if the 'discount_applied' and 'promo_code_used' columns are identical across all rows
(df['discount_applied'] == df['promo_code_used']).all()

True

In [34]:
# Drop the redundant 'promo_code_used' column to optimize the dataset size and avoid multicollinearity
df = df.drop('promo_code_used', axis=1)

In [35]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequeny_days'],
      dtype='object')

### Database Integration (SQLite)
We will export the cleaned dataset to a local SQLite database table. This allows other stakeholders or applications to run SQL queries directly on the structured data.

In [37]:
import sqlite3

In [38]:
# Connect to (or create) a local SQLite database named 'customer_behavior.db'
conn = sqlite3.connect('customer_behavior.db')

# Write the cleaned pandas DataFrame to a SQL table, replacing it if it already exists
df.to_sql('customer_shopping_behavior', conn, if_exists='replace', index=False)

# Always close the database connection when operations are complete
conn.close()

Let's test retrieving data from the newly created database with a simple filter query.

In [40]:
# Re-establish connection to the SQLite database
conn = sqlite3.connect('customer_behavior.db')

# Execute a SQL query to filter for customers over the age of 25 and load the results into a new DataFrame
new_df = pd.read_sql_query("SELECT * FROM customer_shopping_behavior WHERE Age > 25", conn)

# Close the connection
conn.close()